# Run experiments

End-to-end pipeline behind the paper figures. Every step writes to `results/` (git-ignored except
`results/summary/`), so you can run only what you need:

| Step | Output | Needed for |
| --- | --- | --- |
| 0. Preprocess raw CSVs (optional) | `data/trimmed/*.pkl.gz` (already shipped) | everything |
| 1. Base-forecaster metrics | `results/base_forecasters/` | summary |
| 2. Main sweeps (our alg., Hedge-QL, WQL) | `results/omni/`, `results/hedge_ql/`, `results/omni_wql/` | summary |
| 3. US sensitivity runs | `results/sensitivity_us/` | sensitivity summary |
| 4. Build summaries | `results/summary/*.pkl` | `paper_figures.ipynb` |

The same sweeps can be run from the command line, e.g.

```bash
cd src
python run_experiment.py --algo omni_v2 --geos all          # our algorithm, all 52 geos x 4 horizons x c_O sweep
python run_experiment.py --algo ql_pb   --geos all          # Hedge-QL (per-level Hedge) baseline
python run_experiment.py --algo wql     --geos all          # two-player WQL omnipredictor (appendix)
python run_experiment.py --algo omni_v2 --geos US CA --weeks 4 --etas 1.0 --output-dir ../results/my_run
```

In [ ]:
import os, sys, pickle
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), os.pardir, "src")))

import config as cfg
import data_io
import preprocess
import sensitivity as sens
import summary
from run_experiment import run_sweep

GEOS = cfg.GEO_LIST        # all 52 locations (51 states + US-agg)
WEEKS = [1, 2, 3, 4]       # forecast horizons
N_JOBS = -1

## 0. Preprocess raw data (optional)

`data/trimmed/` already holds the preprocessed series. To rebuild them, download
[hospitalizations.zip](https://forecast-eval.s3.us-east-2.amazonaws.com/hospitalizations.zip), unzip the
`hospitalizations_{1,2,3,4}wk.csv` files into `data/hospitalizations/`, and set `REBUILD_DATA = True`.
`impute_na=True` fills missing forecasts with the median of the other forecasters (used by the
algorithms); `impute_na=False` keeps them missing (`*_raw`, used to rank raw base forecasters in Figure 9).

In [ ]:
REBUILD_DATA = False
if REBUILD_DATA:
    preprocess.build_raw_state_pickles()
    for impute_na in (True, False):
        preprocess.build_forecasts_dict(impute_na=impute_na)
        preprocess.build_trimmed(impute_na=impute_na)

## 1. Base-forecaster and ensemble metrics

Omni-error and pinball-loss traces of every base forecaster and ensemble (~1 min per geo).

In [ ]:
from joblib import Parallel, delayed

Parallel(n_jobs=4, verbose=5)(   # each job holds a (T, N, m, F) score tensor (~1 GB for US)
    delayed(preprocess.compute_base_forecaster_metrics)(geo, impute_na=impute_na)
    for geo in GEOS for impute_na in (True, False))

## 2. Main sweeps

Each run is pickled to `results/<algo>/results_wk{w}_{geo}_eta{c}.pkl`; existing files are skipped.
`etas=None` uses the full sweeps in `config` (`ETA_LIST_OMNI` for $c_O$, `ETA_LIST_QL` for $c_H$).
The WQL omnipredictor is not used by any figure.

In [ ]:
run_sweep(algo="omni_v2", geos=GEOS, weeks=WEEKS, etas=None, n_jobs=N_JOBS)   # our algorithm
run_sweep(algo="ql_pb",   geos=GEOS, weeks=WEEKS, etas=None, n_jobs=N_JOBS)   # Hedge-QL

In [ ]:
RUN_WQL = False
if RUN_WQL:
    run_sweep(algo="wql", geos=GEOS, weeks=WEEKS, etas=None, n_jobs=N_JOBS)

## 3. US sensitivity runs

**Runtime** (Figure 5; US-agg, 4-week-ahead, $c_O=1$): grid unit, number of quantile levels, number of
base forecasters. Timings depend on the machine.

**Total QL vs. grid unit** (Figure 8; US-agg, all horizons): both algorithms over their full learning-rate
sweeps at every grid unit in `config.SENSITIVITY_UNITS`. This is the slow part (the finest grid dominates).

In [ ]:
SEED, N_REPEATS = 0, 1
os.makedirs(cfg.SENSITIVITY_DIR, exist_ok=True)

loaded = data_io.load_geo_arrays(cfg.SENSITIVITY_GEO, cfg.SENSITIVITY_WEEK)
runtime = {
    "grid": sens.run_grid_unit_experiment_cached(
        loaded, units=cfg.SENSITIVITY_UNITS, eta_multiplier=cfg.SENSITIVITY_ETA_MULTIPLIER,
        seed=SEED, n_repeats=N_REPEATS),
    "quantile": sens.run_quantile_experiment(
        loaded, k_list=cfg.SENSITIVITY_N_QUANTILES, unit=loaded["unit"],
        eta_multiplier=cfg.SENSITIVITY_ETA_MULTIPLIER, seed=SEED, n_repeats=N_REPEATS * 5),
    "forecaster": sens.run_forecaster_experiment(
        geo=cfg.SENSITIVITY_GEO, week=cfg.SENSITIVITY_WEEK, fracs=cfg.SENSITIVITY_FORECASTER_FRACS,
        unit=loaded["unit"], eta_multiplier=cfg.SENSITIVITY_ETA_MULTIPLIER, seed=SEED,
        n_repeats=N_REPEATS * 5, fix_grid=True),
}
pickle.dump(runtime, open(os.path.join(cfg.SENSITIVITY_DIR, "sensitivity_results.pkl"), "wb"))

In [ ]:
all_weeks = {}
for w in WEEKS:
    loaded_w = data_io.load_geo_arrays(cfg.SENSITIVITY_GEO, w)
    all_weeks[w] = {
        "grid": sens.run_grid_unit_experiment_cached(
            loaded_w, units=cfg.SENSITIVITY_UNITS, eta_multiplier=list(cfg.ETA_LIST_OMNI),
            seed=SEED, n_repeats=N_REPEATS),
        "grid_qlpb": sens.run_grid_unit_qlpb_experiment(
            loaded_w, units=cfg.SENSITIVITY_UNITS, eta_multiplier=list(cfg.ETA_LIST_QL),
            round_Y_F=True, seed=SEED, n_repeats=N_REPEATS),
    }
pickle.dump(all_weeks, open(os.path.join(cfg.SENSITIVITY_DIR, "grid_unit_error_all_weeks.pkl"), "wb"))

## 4. Build the summaries read by `paper_figures.ipynb`

In [ ]:
summary.build_main_summary()
summary.build_sensitivity_summary()